# End-to-end word accuracy: what happens when the letter count is not given

Every word-accuracy figure so far (33% for HOG, 53% for geometry + SVM) was computed on the
pseudo-labelled holds in `hold_landmarks.npy`. Pseudo-labelling is told the word, so it
places exactly as many holds as the word has letters. The decoder then filters the lexicon by
hold count. **Those figures therefore assume the letter count is known.**

A live system has to find the holds itself. This notebook measures that path on the same 218
clips and the same leave-one-signer-out folds:

| | holds come from | letter count |
|---|---|---|
| **oracle** | pseudo-labelling (told the word) | known |
| **free** | `sofi_dynamic.segment_clip`, unaided, default parameters | has to be found |

The letter model is identical in both rows: trained on the other four signers' pseudo-labelled
holds. Only the test-time holds differ.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import evaluation as ev, end_to_end as e2e, word_decode as wd

pd.set_option("display.width", 200)

## 1. Does the extraction code reproduce the cached landmarks?

No script that wrote `hold_landmarks.npy` is in the repo, so before trusting landmarks from
new frames, the pseudo-labelled hold frames are re-detected with `end_to_end.detect_pts` and
compared with the cache. Differences are in pixels.

In [2]:
v = e2e.verify_against_cache(n=150, seed=0)
print(f"{len(v)} holds re-detected, {int(v.xy_px.isna().sum())} not found")
print(v[["xy_px", "z_px"]].describe().loc[["mean", "max"]].round(4).to_string())

150 holds re-detected, 0 not found
      xy_px  z_px
mean    0.0   0.0
max     0.0   0.0


## 2. How often does free segmentation get the count right?

`freeseg_landmarks.npy` / `freeseg_meta.csv` were written by `python end_to_end.py`.

In [3]:
Mf = pd.read_csv(e2e.OUT_META)
P, M = ev.load_dataset()
print(f"free holds: {len(Mf)} in {Mf.groupby(['signer','video']).ngroups} clips; "
      f"no hand found: {(Mf.detected == 0).sum()}; read as left hand: {(Mf.hand == 'Left').sum()}")
print(f"frame used differs from the minimum frame: {(Mf.frame_used != Mf.frame).sum()} holds")

clips, lex = e2e.loso_clips("vote")
C = pd.DataFrame([{"signer": c["signer"], "n_true": c["n_true"], "oracle": len(c["oracle"]),
                   "free": len(c["free"])} for c in clips])
for col in ("oracle", "free"):
    err = C[col] - C.n_true
    print(f"{col:>6}: exact {100*(err == 0).mean():.1f}%   off by one {100*(err.abs() == 1).mean():.1f}%   "
          f"off by 2+ {100*(err.abs() >= 2).mean():.1f}%   MAE {err.abs().mean():.2f}")
print("\nfree count error by signer (MAE):")
print(C.assign(err=(C.free - C.n_true).abs()).groupby("signer").err.mean().round(2).to_string())

free holds: 1362 in 218 clips; no hand found: 1; read as left hand: 15
frame used differs from the minimum frame: 9 holds


oracle: exact 91.3%   off by one 7.8%   off by 2+ 0.9%   MAE 0.10
  free: exact 32.1%   off by one 40.4%   off by 2+ 27.5%   MAE 1.19

free count error by signer (MAE):
signer
signer1-F    1.41
signer2-F    0.96
signer3-F    1.31
signer4-L    1.33
signer5-R    0.73


The current decoder only considers words within one letter of the hold count. A clip whose
count is off by two or more therefore cannot be decoded correctly, whatever the letter model
says.

## 3. Word accuracy, oracle against free

Scores are the one-vs-one vote share the browser computes (`web/svm_classifier.js`), which is
deterministic. The decoder is `word_decode.decode_current`, a port of `web/word_decode.js`.

In [4]:
def report(clips, lex, decoder, label):
    rows = []
    for which in ("oracle", "free"):
        T = e2e.word_table(clips, lex, decoder, which)
        sg = T.signer.to_numpy()
        for k in ("top1", "top3"):
            r = ev.summarise(f"{label} | {which} | {k}", T[k].to_numpy(), sg)
            rows.append(r)
    return pd.DataFrame(rows)

R = report(clips, lex, wd.decode_current, "vote share")
print(R.round(1).to_string(index=False))

                    metric   n  pooled %  mean of folds %  t half-width  boot lo %  boot hi %
vote share | oracle | top1 218      56.0             55.0          25.0       38.1       70.0
vote share | oracle | top3 218      79.4             80.3          19.3       65.0       89.6
  vote share | free | top1 218      23.4             23.6           8.5       18.0       29.1
  vote share | free | top3 218      33.5             34.5          11.9       26.2       39.8


In [5]:
T = e2e.word_table(clips, lex, wd.decode_current, "free")
To = e2e.word_table(clips, lex, wd.decode_current, "oracle")
per = pd.DataFrame({"clips": T.groupby("signer").size(),
                    "oracle top-1 %": 100 * To.groupby("signer").top1.mean(),
                    "free top-1 %": 100 * T.groupby("signer").top1.mean(),
                    "free top-3 %": 100 * T.groupby("signer").top3.mean()})
print(per.round(1).to_string())

           clips  oracle top-1 %  free top-1 %  free top-3 %
signer                                                      
signer1-F     49            61.2          18.4          32.7
signer2-F     49            71.4          32.7          38.8
signer3-F     45            73.3          15.6          20.0
signer4-L     49            26.5          24.5          34.7
signer5-R     26            42.3          26.9          46.2


## 4. Where the loss comes from

In [6]:
T["count error"] = (T.n_holds - T.n_true).clip(-2, 2).map({-2: "-2 or worse", -1: "-1", 0: "0", 1: "+1", 2: "+2 or worse"})
order = ["-2 or worse", "-1", "0", "+1", "+2 or worse"]
print((T.groupby("count error").agg(clips=("top1", "size"), top1=("top1", "mean"), top3=("top3", "mean"))
        .reindex(order).assign(top1=lambda d: (100 * d.top1).round(1), top3=lambda d: (100 * d.top3).round(1))).to_string())

             clips  top1  top3
count error                   
-2 or worse     29   0.0   0.0
-1              38   2.6  15.8
0               70  70.0  88.6
+1              50   2.0  10.0
+2 or worse     31   0.0   0.0

## 5. Same comparison with Platt probabilities, five seeds

The notebook `scarce_data_recognition.ipynb` used `predict_proba`. Platt scaling is fitted on
random internal folds, so the word figures move with the seed.

In [7]:
rows = []
for seed in range(5):
    cl, _ = e2e.loso_clips("proba", seed=seed)
    row = {"seed": seed}
    for which in ("oracle", "free"):
        Tw = e2e.word_table(cl, lex, wd.decode_current, which)
        row[f"{which} top-1"] = 100 * Tw.groupby("signer").top1.mean().mean()
        row[f"{which} top-3"] = 100 * Tw.groupby("signer").top3.mean().mean()
    rows.append(row)
S = pd.DataFrame(rows).set_index("seed")
print(S.round(1).to_string())
print("\nmean over seeds:", S.mean().round(1).to_dict())
print("range over seeds:", (S.max() - S.min()).round(1).to_dict())

      oracle top-1  oracle top-3  free top-1  free top-3
seed                                                    
0             53.1          75.3        24.9        41.2
1             53.1          75.7        24.9        40.0
2             53.4          75.3        23.7        40.4
3             55.1          75.3        24.9        40.4
4             53.0          74.5        24.9        41.2

mean over seeds: {'oracle top-1': 53.5, 'oracle top-3': 75.2, 'free top-1': 24.7, 'free top-3': 40.7}
range over seeds: {'oracle top-1': 2.1, 'oracle top-3': 1.2, 'free top-1': 1.2, 'free top-3': 1.2}


## What this means

* The published-style figure (53 to 55% top-1) is an **upper bound that assumes perfect
  segmentation**. Measured end to end with the segmenter the project actually has, top-1 is
  the "free" row above.
* The letter model is the same in both rows. The whole gap is the hold count: free
  segmentation is exact on about a third of clips, and the decoder cannot recover from a
  missing or extra hold because it compares hold *i* with letter *i*.
* Clips with the right count decode almost as well as the oracle. So the decoder, not the
  letter model, is the cheapest thing to fix next: one that tolerates a missed or spurious
  hold (`alignment_decoder.ipynb`).
* This uses offline Algorithm 2. The browser runs the causal version with extra gates, which
  has not been measured on words.